# Bayes Factor: Model Comparison

This notebook trains a classifier to distinguish spectra from competing
atmospheric models and computes the Bayes factor on the real observation.

The classifier reuses the **frozen embedding** from the trained NPE estimator,
so it sees the same representation the retrieval network learned.

**Example:** cloudfree vs cloudy

## The Likelihood Ratio Trick

The Bayes factor between two models $M_A$ and $M_B$ is:

$$B_{AB} = \frac{p(x_{\mathrm{obs}} | M_A)}{p(x_{\mathrm{obs}} | M_B)} = \frac{\int p(x_{\mathrm{obs}} | \theta, M_A)\, p(\theta | M_A)\, d\theta}{\int p(x_{\mathrm{obs}} | \theta, M_B)\, p(\theta | M_B)\, d\theta}$$

Computing these marginal likelihoods (evidences) directly is expensive.
The **likelihood ratio trick** avoids this entirely.

### Key insight

Train a binary classifier $d(x)$ to distinguish simulated data from $M_A$ vs $M_B$.
If the classifier is trained with equal class priors $p(M_A) = p(M_B) = 0.5$ and converges
to the Bayes-optimal decision boundary, then its output satisfies:

$$d^*(x) = \frac{p(x | M_B)}{p(x | M_A) + p(x | M_B)}$$

The log-odds of the classifier directly give the **log Bayes factor**:

$$\log B_{AB} = \log\frac{p(x|M_A)}{p(x|M_B)} = \log\frac{1 - d^*(x)}{d^*(x)} = -\mathrm{logit}(d^*(x))$$

### In practice

1. Sample $\theta \sim p(\theta|M_A)$, simulate $x \sim p(x|\theta, M_A)$ → label 0
2. Sample $\theta \sim p(\theta|M_B)$, simulate $x \sim p(x|\theta, M_B)$ → label 1
3. Train a classifier $d_\phi(x)$ with BCE loss on these pairs
4. Evaluate on $x_{\mathrm{obs}}$: the logit output approximates $\log B_{BA}$

### Why this works

- The BCE loss's minimizer is exactly the density ratio $p(x|M_B) / (p(x|M_A) + p(x|M_B))$
- No need to estimate the intractable evidences separately
- The classifier implicitly marginalizes over $\theta$ through the training data
- Works for any simulator — no likelihood function needed

### Multi-class extension

For $N > 2$ models, train with cross-entropy loss. The softmax outputs give:

$$p(M_k | x) = \frac{p(x | M_k)}{\sum_j p(x | M_j)}$$

Pairwise Bayes factors follow: $B_{ij} = p(M_i | x) / p(M_j | x)$

### References

- Hermans, Begy & Louppe (2020): *Likelihood-free MCMC with Amortized Approximate Ratio Estimators*
- Cranmer, Brehmer & Louppe (2020): *The frontier of simulation-based inference*

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.utils.checkpoint import load_checkpoint, load_model_state
from sbi4atmret.evaluation.bayes_factor import (
    BayesFactorClassifier,
    prepare_classification_data,
    train_bayes_classifier,
    compute_bayes_factor,
)

## 1. Load Config & Trained NPE Model

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"
with open(config_path) as f:
    config = BaseConfig(**yaml.safe_load(f))

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build model and load checkpoint
model = BaseModel(config).build()
checkpoint_path = Path("path/to/states_800.pth")
checkpoint = load_checkpoint(checkpoint_path, device)
load_model_state(model.estimator, checkpoint)
model.estimator.to(device).eval()

# The embedding we'll reuse (frozen)
embedding = model.estimator.embedding
print(f"Embedding loaded. Output dim needs to be determined from architecture.")

## 2. Load Spectra from Competing Models

Load pre-simulated spectra from the cloudfree and cloudy models.
These should already be processed through the pipe (same as NPE training input).

In [ ]:
# Example: load processed spectra from your datasets
# These are the merged x vectors (same format as NPE training)
#
# Option 1: Use your existing DataLoaders
# from sbi4atmret.datasets.DatasetBase import Dataset
# dataset = Dataset(config)
# ... iterate dataloaders and collect processed x ...
#
# Option 2: Load pre-saved tensors
# spectra_cloudfree = torch.load("path/to/cloudfree_spectra.pt")
# spectra_cloudy = torch.load("path/to/cloudy_spectra.pt")

# Placeholder dimensions:
# spectra_cloudfree: (N, D_obs) — e.g., (50000, 1732)
# spectra_cloudy: (N, D_obs)

print(f"Cloudfree spectra: {spectra_cloudfree.shape}")
print(f"Cloudy spectra: {spectra_cloudy.shape}")

## 3. Prepare Data

In [ ]:
# Prepare multi-class data
# Add more models here if needed: {"cloudfree": ..., "cloudy": ..., "patchy": ...}
spectra_dict = {
    "cloudfree": spectra_cloudfree,
    "cloudy": spectra_cloudy,
}

x_all, labels_all, model_names = prepare_classification_data(spectra_dict)

print(f"Total samples: {len(x_all)}")
print(f"Models: {model_names}")
print(f"Label distribution: {[(labels_all == i).sum().item() for i in range(len(model_names))]}")

# Train/val split
n_val = int(0.1 * len(x_all))
train_x, val_x = x_all[n_val:], x_all[:n_val]
train_labels, val_labels = labels_all[n_val:], labels_all[:n_val]

print(f"Train: {len(train_x)}, Val: {len(val_x)}")

## 4. Build & Train Classifier

In [ ]:
# Determine embedding output dim by running a dummy forward pass
with torch.no_grad():
    dummy = torch.randn(1, train_x.shape[-1]).to(device)
    emb_dim = embedding(dummy).shape[-1]

print(f"Embedding output dim: {emb_dim}")

classifier = BayesFactorClassifier(
    embedding=embedding,
    embedding_dim=emb_dim,
    n_classes=len(model_names),
    head_hidden=[128, 64],
    dropout=0.1,
)

# Count trainable params (only the head)
n_trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in classifier.parameters() if not p.requires_grad)
print(f"Trainable: {n_trainable:,} | Frozen (embedding): {n_frozen:,}")

In [ ]:
save_path = Path("bf_classifier_best.pth")

history = train_bayes_classifier(
    classifier,
    train_x, train_labels,
    val_x, val_labels,
    n_epochs=1024,
    batch_size=1024,
    lr=1e-3,
    patience=32,
    device=device,
    save_path=save_path,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.legend()
ax1.set_title("Loss")

ax2.plot(history["val_accuracy"], color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title(f"Val Accuracy (final: {history['val_accuracy'][-1]:.3f})")
ax2.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.show()

## 5. Compute Bayes Factor on Real Observation

In [ ]:
# Load your real observation (same format as training x)
# x_obs = torch.from_numpy(observation.full_observation).unsqueeze(0).float()

result = compute_bayes_factor(
    classifier, x_obs, model_names=model_names, device=device
)

print("\n=== Bayes Factor Results ===")
print(f"\nModel probabilities:")
for name, prob in result.probabilities.items():
    print(f"  {name}: {prob:.4f} ({prob*100:.1f}%)")

print(f"\nPairwise Bayes factors:")
for pair, interp in result.interpretations.items():
    print(f"  {pair[0]} vs {pair[1]}: {interp}")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(model_names, [result.probabilities[n] for n in model_names],
              color=["steelblue", "darkorange", "seagreen"][:len(model_names)])
ax.set_ylabel("Posterior model probability")
ax.set_title("Model Comparison — Real Observation")
ax.set_ylim(0, 1)

for bar, name in zip(bars, model_names):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{result.probabilities[name]:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 6. Validation: Check on Known Samples

In [ ]:
# Test on a few known cloudfree and cloudy spectra
classifier.eval()

with torch.no_grad():
    # First 10 from each class in validation
    cf_probs = classifier.predict_proba(val_x[:10].to(device)).cpu().numpy()
    cl_idx = (val_labels[:100] == 1).nonzero()[:10].flatten()
    cl_probs = classifier.predict_proba(val_x[cl_idx].to(device)).cpu().numpy()

print("Known cloudfree samples → prob(cloudfree):")
print(f"  Mean: {cf_probs[:, 0].mean():.3f} (should be ~1)")

print(f"\nKnown cloudy samples → prob(cloudy):")
print(f"  Mean: {cl_probs[:, 1].mean():.3f} (should be ~1)")